# Map renderer notebook

This version renders maps using local Natural Earth data. Urban areas are hatched at every zoom level. Land, coastlines, water and city labels are retained, with larger roads from the global Natural Earth road network. 

Installing Node packages requires internet access on the first run; rendering maps afterwards does not. WebP files are written directly to `map-tiles/trips`, `map-tiles/1y` and `map-tiles/5y`, replacing maps with the same filenames from the other renderer.


In [ ]:
from pathlib import Path

# Start the notebook from dataProcessing/1_mapExporter or the project folder.
PACKAGE_ROOT = Path.cwd().resolve()
if PACKAGE_ROOT.name == 'dataProcessing':
    PACKAGE_ROOT = PACKAGE_ROOT / '1_mapExporter'
elif PACKAGE_ROOT.name != '1_mapExporter':
    PACKAGE_ROOT = PACKAGE_ROOT / 'dataProcessing/1_mapExporter'
import os
import shutil
import subprocess

# If the Python packages are missing, run this separately once:
# %pip install tqdm pillow

NODE = None
nodeCandidates = [shutil.which('node'), '/opt/homebrew/bin/node', '/usr/local/bin/node']
for candidate in nodeCandidates:
    if candidate is None or Path(candidate).exists() == False:
        continue
    version = subprocess.run([candidate, '-p', 'process.versions.node'], capture_output=True, text=True)
    if version.returncode == 0 and tuple(map(int, version.stdout.strip().split('.'))) >= (22, 12, 0):
        NODE = candidate
        break
if NODE is None:
    raise RuntimeError('Please install Node.js 22.12 or newer and restart Jupyter.')

# Jupyter may not inherit the terminal PATH. npm also needs to find node.
nodeEnvironment = dict(os.environ)
nodeEnvironment['PATH'] = str(Path(NODE).parent) + os.pathsep + nodeEnvironment.get('PATH', '')
NPM = shutil.which('npm', path=nodeEnvironment['PATH'])
if NPM is None:
    raise RuntimeError('npm was not found. Please check your Node.js installation.')

# An interrupted installation can also leave a node_modules folder behind.
packageCheck = subprocess.run(
    [NODE, '--input-type=module', '-e',
     "await Promise.all(['d3', 'topojson-client', '@resvg/resvg-js', 'sharp'].map(name => import(name)))"],
    cwd=PACKAGE_ROOT, env=nodeEnvironment, capture_output=True,
)
if packageCheck.returncode == 0:
    print('Node packages are already installed.')
else:
    print('Installing Node packages from package-lock.json …')
    subprocess.run([NPM, 'ci'], cwd=PACKAGE_ROOT, env=nodeEnvironment, check=True)


## Settings

In [ ]:
PROJECT_ROOT = PACKAGE_ROOT.parent.parent
DATA_CSV = PROJECT_ROOT / 'data.csv'
HIERARCHY_JS = PROJECT_ROOT / 'assets/js/hierarchy-transform.js'
MAP_DIR = PACKAGE_ROOT / 'data/maps'
OUTPUT_ROOT = PROJECT_ROOT / 'map-tiles'

# System fonts: serif (Times), sans-serif (Arial).
FONT_FAMILY = 'Times New Roman'
FONT_STYLE = 'italic'
FONT_WEIGHT = 400
FONT_FILES = []
FONT_DIRS = [Path.home() / 'Library/Fonts', Path('/Library/Fonts'), Path('/System/Library/Fonts')]
LOAD_SYSTEM_FONTS = True
DEFAULT_FONT_FAMILY = FONT_FAMILY

# These modes are defined in assets/js/hierarchy-transform.js
VARIANTS = ['trips', '5y', '1y']
WIDTH = 260                  # tileViewBoxSize in assets/js/main.js 
HEIGHT = WIDTH
SCALE = 5                      # 260 × 5 = 1300 pixels
WEBP_QUALITY = 90
WEBP_EFFORT = 5                # 0 = fastest, 6 = smallest files; 5 works well for batch exports
PADDING = 50
SINGLE_SPAN_DEG = 0.6          # Reference span for a single location
MIN_ITEMS_PER_TILE = 1         # 1 = all tiles; 2 excludes tiles containing only one photo
SKIP_EXISTING = False          # False = render every tile again during a full queue run
START_AT = 0
STOP_BEFORE = None             # e.g. 10; None = all tiles
PREVIEW_EVERY = 1              # 0 disables previews

# Default colors and line styles
STYLE = {
    'background': '#000000',
    'ocean': '#1a1e1e',
    'land': '#767676',
    'graticule': '#303030', 'graticuleWidth': 0.3,
    'urban': '#ffffff', 'urbanOpacity': 0.4, 'urbanLineWidth': 0.8,
    'urbanSpacing': 4, 'urbanAngle': 45,
    'showRoads': True, 'roads': '#d4d4d4', 'roadWidth': 0.45, 'roadOpacity': 0.75,
    'water': '#1a1e1e',
    'river': '#1a1e1e',
    'coastline': '#4a4a4a', 'coastlineWidth': 0.4, 'coastlineOpacity': 0.9,
    # Natural Earth city labels
    'showCities': True, 'cityColor': '#eeeeee',
    'citySize': 9, 'cityFallbackSize': 10,
    'cityFamily': FONT_FAMILY, 'cityWeight': FONT_WEIGHT, 'cityStyle': FONT_STYLE,
    'cityHalo': '#767676', 'cityHaloWidth': 2.2, 'cityHaloOpacity': 1,
    'cityMaxPrimary': 8, 'cityMaxRelaxed': 14,
}


## Start the worker and load the queue

In [ ]:
# This notebook is a translation of D3 map generation into a notebook to match D3s map projections with a static generated map file
import json, subprocess, shutil
from IPython.display import display
from PIL import Image as PillowImage
from tqdm.auto import tqdm

class D3Worker:
    def __init__(self):
        worker_env = dict(
            nodeEnvironment,
            TILE_PACKAGE_ROOT=str(PACKAGE_ROOT), DATA_CSV=str(DATA_CSV),
            HIERARCHY_JS=str(HIERARCHY_JS), MAP_DIR=str(MAP_DIR),
        )
        self.proc = subprocess.Popen(
            [NODE, str(PACKAGE_ROOT / 'render-worker.mjs')], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
            stderr=subprocess.PIPE, text=True, bufsize=1, cwd=PACKAGE_ROOT, env=worker_env,
        )
        while True:
            event = self._read()
            if event['type'] == 'loading': print(event['message'])
            if event['type'] == 'ready':
                self.tiles = event['tiles']
                break

    def _read(self):
        line = self.proc.stdout.readline()
        if not line:
            error = self.proc.stderr.read()
            raise RuntimeError(f'The D3 worker stopped.\n{error}')
        return json.loads(line)

    def render(self, index, **settings):
        command = {'action': 'render', 'index': int(index), **settings}
        self.proc.stdin.write(json.dumps(command) + '\n')
        self.proc.stdin.flush()
        events = []
        while True:
            event = self._read(); events.append(event)
            if event['type'] in ('done', 'failed'): return event, events

    def configure(self, mode):
        command = {'action': 'configure', 'mode': mode, 'minItemsPerTile': MIN_ITEMS_PER_TILE}
        self.proc.stdin.write(json.dumps(command) + '\n')
        self.proc.stdin.flush()
        event = self._read()
        if event['type'] != 'configured': raise RuntimeError(event.get('message', event))
        self.tiles = event['tiles']
        return event

    def close(self):
        if self.proc.poll() is None:
            self.proc.stdin.write(json.dumps({'action': 'quit'}) + '\n')
            self.proc.stdin.flush(); self.proc.wait(timeout=10)

worker = D3Worker()
initial_info = worker.configure(VARIANTS[0])
print(f"Worker ready; {VARIANTS[0]} contains {len(initial_info['tiles'])} tiles with MIN_ITEMS_PER_TILE={MIN_ITEMS_PER_TILE}")


## Render the queue sequentially

In [ ]:
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
try:
  for variant in VARIANTS:
    info = worker.configure(variant)
    output_dir = OUTPUT_ROOT / variant
    output_dir.mkdir(parents=True, exist_ok=True)
    tiles = info['tiles']
    end = len(tiles) if STOP_BEFORE is None else min(STOP_BEFORE, len(tiles))
    indices = list(range(START_AT, end))
    settings = dict(
        outputDir=str(output_dir), width=WIDTH, height=HEIGHT,
        scale=SCALE, webpQuality=WEBP_QUALITY, webpEffort=WEBP_EFFORT,
        padding=PADDING, singleSpanDeg=SINGLE_SPAN_DEG, style=STYLE,
        fontFiles=[str(path) for path in FONT_FILES], loadSystemFonts=LOAD_SYSTEM_FONTS,
        fontDirs=[str(path) for path in FONT_DIRS if path.exists()],
        defaultFontFamily=DEFAULT_FONT_FAMILY,
    )
    progress = tqdm(indices, unit='tile', desc=variant)
    for index in progress:
        tile = tiles[index]
        stem = f"{int(tile['originalIndex']):04d}"
        webp_file = output_dir / f'{stem}.webp'
        if SKIP_EXISTING and webp_file.exists():
            progress.set_postfix_str(f'{stem} skipped')
            continue

        progress.set_postfix_str(f"{stem} · {tile['name']}")
        result, events = worker.render(index, stem=stem, **settings)
        warnings = [e['message'] for e in events if e['type'] == 'warning']
        for warning in warnings:
            print(f"{tile['name']}: {warning}")
        if result['type'] == 'failed':
            raise RuntimeError(result['message'])
        if PREVIEW_EVERY and (index - START_AT) % PREVIEW_EVERY == 0:
            display(PillowImage.open(webp_file).resize((300, 300)))
finally:
    worker.close()

print(f'Output: {OUTPUT_ROOT}')